<a href="https://colab.research.google.com/github/keshav123333/Pytorch_beginners/blob/main/random_log_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [ ]:



class LogLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()

        # unconstrained parameters
        self.raw_base = nn.Parameter(
            torch.randn(out_features, in_features)
        )
        self.linear1=nn.Parameter(torch.randn(out_features,in_features))

        self.linear2=nn.Linear(in_features,out_features)
        self.bias = nn.Parameter(
            torch.zeros(out_features)
        )


    def forward(self, x):

        base = 1.1 + F.softplus(self.raw_base)

        eps = 1e-6
        ln_x = (
            torch.sign(x)
            * torch.log1p(torch.abs(x))
        ).unsqueeze(1)

        ln_base = torch.log(base)

        out = ln_x / ln_base
        out=self.linear1*out



        # sum all input features
        out = out.sum(dim=-1)
        second_out=self.linear2(x)
        out=out+second_out
        out = out + self.bias

        return out

In [ ]:
class DemoModel(nn.Module):
  def __init__(self,input_size):
    super().__init__()
    self.layers=nn.Sequential(
        LogLayer(input_size,128),
        nn.LayerNorm(128),
        nn.ReLU(),
        nn.Dropout(0.3),

        LogLayer(128,32),
        nn.LayerNorm(32),


        nn.ReLU(),

        nn.Dropout(0.2),
        LogLayer(32,1),


    )
  def forward(self,x):
    return self.layers(x)


In [ ]:
torch.manual_seed(0)
x1=torch.randint(1,10,size=(10000,2) );
x1=x1.to(torch.float32)
weights=torch.randn(2,1)
y = (
    x1 @ weights
    + torch.log(x1).sum(dim=-1, keepdim=True)

)

In [ ]:
x_val=torch.randint(1,10,size=(1000,2) );
x_val=x_val.to(torch.float32)
y_val=(x_val@weights++ torch.log(x_val).sum(dim=-1, keepdim=True))


In [ ]:
x1

tensor([[9., 1.],
        [3., 7.],
        [8., 7.],
        ...,
        [2., 1.],
        [5., 3.],
        [9., 6.]])

In [ ]:
y

tensor([[-0.6036],
        [ 2.7667],
        [ 2.1368],
        ...,
        [ 0.1472],
        [ 1.3925],
        [ 1.6800]])

In [ ]:
model = DemoModel(2).to(device)

In [ ]:
x1 = x1.to(device)
y = y.to(device)

x_val = x_val.to(device)
y_val = y_val.to(device)

In [ ]:


optimizer=torch.optim.Adam(params= model.parameters(),lr=0.01)
loss=nn.MSELoss()

In [ ]:
for i in range(10000):
    model.train()

    optimizer.zero_grad()

    y_pred = model(x1)
    train_loss = loss(y_pred, y)

    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_pred = model(x_val)
        val_loss = loss(val_pred, y_val)

    print(
        f"Epoch {i+1:3d} | "
        f"train_loss={train_loss.item():.6f} | "
        f"val_loss={val_loss.item():.6f}"
    )

Streaming output truncated to the last 5000 lines.
Epoch 5001 | train_loss=0.020993 | val_loss=0.150544
Epoch 5002 | train_loss=0.020120 | val_loss=0.145427
Epoch 5003 | train_loss=0.020617 | val_loss=0.152569
Epoch 5004 | train_loss=0.020505 | val_loss=0.152302
Epoch 5005 | train_loss=0.020599 | val_loss=0.141619
Epoch 5006 | train_loss=0.020744 | val_loss=0.145883
Epoch 5007 | train_loss=0.019176 | val_loss=0.160803
Epoch 5008 | train_loss=0.020565 | val_loss=0.150178
Epoch 5009 | train_loss=0.020795 | val_loss=0.145645
Epoch 5010 | train_loss=0.020345 | val_loss=0.152621
Epoch 5011 | train_loss=0.019656 | val_loss=0.143772
Epoch 5012 | train_loss=0.020078 | val_loss=0.142520
Epoch 5013 | train_loss=0.020867 | val_loss=0.157451
Epoch 5014 | train_loss=0.020606 | val_loss=0.152224
Epoch 5015 | train_loss=0.019573 | val_loss=0.138843
Epoch 5016 | train_loss=0.020439 | val_loss=0.146312
Epoch 5017 | train_loss=0.020816 | val_loss=0.153424
Epoch 5018 | train_loss=0.020892 | val_loss=0.14

In [ ]:
x = torch.tensor([[4.0, 9.0]])

with torch.no_grad():
    y = model(x)

print(y)

tensor([[5.6524]])


In [ ]:
y[:12]

tensor([[ 0.7193],
        [ 3.3973],
        [ 2.2395],
        [-0.1900],
        [ 0.1826],
        [ 0.4041],
        [ 1.2800],
        [ 3.0643],
        [-0.4049],
        [ 1.2752],
        [ 2.0898],
        [ 0.7206]], device='cuda:0')